# Visual Learning session metadata

Builds `visual_learning_session_metadata.csv` — one row per session for six mice.

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

DATA_DIR = '/data'
OUTPUT_DIR = '/data/metadata'
CAPSULE_MOUNT = 'Visual-Learning-SWDB' 


In [2]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
OUTPUT_DIR = '/data/metadata'
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
   version="v2",
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

https://api.allenneuraldynamics.org/v2/metadata_index/data_assets


In [3]:
# The cohort is defined by subject, not by project_name -- several project_name
# values are interleaved across the same six mice.
CTL_MICE = ['782149', '790322', '788406', '800792', '800995', '804363']

## Query

The per-plane fields are nested three deep
(`data_streams[] -> configurations[] -> images[] -> planes[]`), so the two `$reduce`
stages flatten that to one `planes` array per asset before we take its size and pluck
the depth / structure / index off it. Flattening server-side keeps the response small
— we never pull the full imaging config.

In [4]:
aggregate = [
  {
    "$match": {
      "data_description.subject_id": {"$in": CTL_MICE},
      "name": {"$regex": "^multiplane-ophys_"},
      "acquisition.acquisition_type": {"$exists": True},
    },
  },
  {
    "$project": {
      "name": 1,
      "subject_id": "$data_description.subject_id",
      "project_name": "$data_description.project_name",
      "acquisition_type": "$acquisition.acquisition_type",
      "session_start_time": "$acquisition.acquisition_start_time",
      "session_end_time": "$acquisition.acquisition_end_time",
      "rig": "$acquisition.instrument_id",
      "genotype": "$subject.subject_details.genotype",
      "sex": "$subject.subject_details.sex",
      "date_of_birth": "$subject.subject_details.date_of_birth",
      # flatten data_streams[] -> configurations[] -> images[] -> planes[]
      "planes": {"$reduce": {
          "input": {"$reduce": {
              "input": "$acquisition.data_streams", "initialValue": [],
              "in": {"$concatArrays": [
                  "$$value", {"$ifNull": ["$$this.configurations", []]}]}}},
          "initialValue": [],
          "in": {"$concatArrays": ["$$value",
              {"$reduce": {
                  "input": {"$ifNull": ["$$this.images", []]}, "initialValue": [],
                  "in": {"$concatArrays": [
                      "$$value", {"$ifNull": ["$$this.planes", []]}]}}}]}}},
    }
  },
  {
    "$project": {
      "name": 1, "subject_id": 1, "project_name": 1, "acquisition_type": 1,
      "session_start_time": 1, "session_end_time": 1, "rig": 1,
      "genotype": 1, "sex": 1, "date_of_birth": 1,
      "n_planes": {"$size": "$planes"},
      "plane_indices": "$planes.plane_index",
      "imaging_depths": "$planes.depth",
      "targeted_structures": "$planes.targeted_structure.acronym",
    }
  },
]

records = docdb_api_client.aggregate_docdb_records(
    pipeline = aggregate,
)
print(f'{len(records)} assets')

if len(records) == 0:
    raise RuntimeError('No assets matched -- check the client version and pipeline.')

1879 assets


## Session table

One row per session, keeping only the newest `_processed_` generation: a session is
reprocessed whenever the pipeline changes, so it appears several times under different
stamps. Anchoring the pattern at end-of-string also drops the further-derived assets
(`cortical-zstack-registration`, coreg) that carry `_processed_` mid-name.

In [5]:
PROCESSED_PATTERN = (r'^multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+'
                     r'_processed_\d{4}-\d{2}-\d{2}_[\d-]+$')

sessions = pd.DataFrame(records)
sessions = sessions[sessions.name.str.match(PROCESSED_PATTERN)].copy()

sessions['session_id'] = sessions.name.str.extract(
    r'^(multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+)_processed_')
sessions['processed_stamp'] = sessions.name.str.extract(
    r'_processed_(\d{4}-\d{2}-\d{2}_[\d-]+)$')

sessions = (sessions.sort_values('processed_stamp')
                    .drop_duplicates('session_id', keep='last'))
print(f'{len(sessions)} unique sessions across {sessions.subject_id.nunique()} mice')

158 unique sessions across 6 mice


In [6]:
# acquisition_date comes off the asset name; session_date/time off the timestamp.
# They agree on every row today -- kept separate because the name is what the mount
# and every derived asset are keyed by.
sessions['acquisition_date'] = sessions.session_id.str.extract(r'_(\d{4}-\d{2}-\d{2})_')
sessions['session_date'] = sessions.session_start_time.map(
    lambda x: datetime.fromisoformat(x).date())
sessions['session_time'] = sessions.session_start_time.map(
    lambda x: datetime.fromisoformat(x).time())
sessions['date_of_birth'] = sessions.date_of_birth.map(
    lambda x: datetime.strptime(x, '%Y-%m-%d').date() if isinstance(x, str) else x)
sessions['age_days'] = [(a - b).days if pd.notnull(b) else np.nan
                        for a, b in zip(pd.to_datetime(sessions.acquisition_date).dt.date,
                                        sessions.date_of_birth)]

sessions['session_type'] = sessions.acquisition_type
sessions['stage'] = sessions.session_type.str.extract(
    r'^(TRAINING_\d|OPHYS_\d|STAGE_\d)')
sessions['image_set'] = sessions.session_type.str.extract(r'_images_([AB])')

sessions = sessions.sort_values(['subject_id', 'acquisition_date'])
sessions['session_number'] = sessions.groupby('subject_id').cumcount() + 1

# Plane columns, ordered by plane_index so depths line up with names
sessions['plane_names'] = [
    [f'{s}_{i}' for i, s in sorted(zip(r.plane_indices, r.targeted_structures))]
    for r in sessions.itertuples()]
sessions['imaging_depths'] = [
    [d for _, d in sorted(zip(r.plane_indices, r.imaging_depths))]
    for r in sessions.itertuples()]
sessions['targeted_structures'] = [
    sorted(set(r.targeted_structures)) for r in sessions.itertuples()]

order = ['subject_id', 'session_id', 'name', 'session_type', 'acquisition_type',
         'stage', 'image_set', 'session_number', 'acquisition_date', 'session_date',
         'session_time', 'age_days', 'genotype', 'sex', 'date_of_birth', 'rig',
         'project_name', 'n_planes', 'plane_names', 'imaging_depths',
         'targeted_structures', 'processed_stamp', '_id']

sessions = sessions[order].reset_index(drop=True)
sessions

,subject_id,session_id,name,session_type,acquisition_type,stage,image_set,session_number,acquisition_date,session_date,...,sex,date_of_birth,rig,project_name,n_planes,plane_names,imaging_depths,targeted_structures,processed_stamp,_id
0,782149,multiplane-ophys_782149_2025-03-25_09-46-08,multiplane-ophys_782149_2025-03-25_09-46-08_pr...,TRAINING_0_gratings_autorewards_15min,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,2025-03-25,...,Male,2024-12-07,422_MESO2_20241017,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[40, 320, 80, 280, 120, 240, 160, 200]",[VISp],2026-08-19_00-32-51,aca6e6d2-9f33-4ed6-8a66-86d69a282c30
1,782149,multiplane-ophys_782149_2025-03-28_10-55-25,multiplane-ophys_782149_2025-03-28_10-55-25_pr...,TRAINING_1_gratings,TRAINING_1_gratings,TRAINING_1,NaN,2,2025-03-28,2025-03-28,...,Male,2024-12-07,429_MESO1_20241016,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 114, 244, 80, 280, 40, 310]",[VISp],2026-08-19_00-34-09,2a3f8254-9f86-42fd-b4d7-fe1877ebf959
2,782149,multiplane-ophys_782149_2025-03-29_10-10-29,multiplane-ophys_782149_2025-03-29_10-10-29_pr...,TRAINING_1_gratings,TRAINING_1_gratings,TRAINING_1,NaN,3,2025-03-29,2025-03-29,...,Male,2024-12-07,429_MESO1_20241016,Learning mFISH-V1omFISH,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[158, 198, 114, 246, 80, 276, 43, 306]",[VISp],2026-08-19_00-33-54,7591b588-f6a1-474d-b00a-3dd10ffb4a60
3,782149,multiplane-ophys_782149_2025-03-31_12-23-33,multiplane-ophys_782149_2025-03-31_12-23-33_pr...,TRAINING_1_gratings,TRAINING_1_gratings,TRAINING_1,NaN,4,2025-03-31,2025-03-31,...,Male,2024-12-07,429_MESO1_20241016,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 115, 245, 80, 280, 40, 310]",[VISp],2026-08-19_00-34-28,d00bf70e-41c7-4ec5-ac76-8aecb010fbe1
4,782149,multiplane-ophys_782149_2025-04-01_09-42-11,multiplane-ophys_782149_2025-04-01_09-42-11_pr...,TRAINING_2_gratings_flashed,TRAINING_2_gratings_flashed,TRAINING_2,NaN,5,2025-04-01,2025-04-01,...,Male,2024-12-07,429_MESO1_20241016,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 115, 240, 85, 280, 40, 300]",[VISp],2026-08-19_00-33-58,ac0f8e4d-cd12-453a-8d83-6b7951a6a957
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,804363,multiplane-ophys_804363_2025-09-08_09-24-39,multiplane-ophys_804363_2025-09-08_09-24-39_pr...,STAGE_1,STAGE_1,STAGE_1,NaN,18,2025-09-08,2025-09-08,...,Female,2025-04-27,429_MESO1_20241016,Learning mFISH-V1omFISH,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 198, 120, 240, 82, 276, 42, 320]",[VISp],2026-08-19_01-05-32,85a8228f-f0b4-480b-a402-95a0379c3c3b
154,804363,multiplane-ophys_804363_2025-09-09_11-44-14,multiplane-ophys_804363_2025-09-09_11-44-14_pr...,STAGE_1,STAGE_1,STAGE_1,NaN,19,2025-09-09,2025-09-09,...,Female,2025-04-27,429_MESO1_20241016,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 122, 240, 84, 280, 44, 322]",[VISp],2026-08-19_01-05-33,385b41e7-68c2-4c0f-8f5c-8771a9413498
155,804363,multiplane-ophys_804363_2025-09-10_14-44-56,multiplane-ophys_804363_2025-09-10_14-44-56_pr...,STAGE_1,STAGE_1,STAGE_1,NaN,20,2025-09-10,2025-09-10,...,Female,2025-04-27,429_MESO1_20241016,Learning mFISH-V1omFISH,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 124, 240, 84, 280, 44, 322]",[VISp],2026-08-19_01-05-12,5ed91986-57b6-43b1-8df1-5d0a39b085d2
156,804363,multiplane-ophys_804363_2025-09-24_13-00-27,multiplane-ophys_804363_2025-09-24_13-00-27_pr...,CENTER_MOUSEMOTION,CENTER_MOUSEMOTION,NaN,NaN,21,2025-09-24,2025-09-24,...,Female,2025-04-27,429_MESO1_20241016,Learning mFISH-V1omFISH,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[163, 206, 123, 246, 85, 290, 47, 328]",[VISp],2026-08-19_01-05-12,df78c027-66d4-4622-81ee-e2c6444df90a


## Restrict to what is actually attached

docDB knows about every processed asset; the capsule only mounts some of them. The
problem sets can only open a file that is on `/data`, so filter both tables to the
assets present in the mount &mdash

If the mount is not attached this cell says so and leaves the tables unfiltered,
rather than silently emitting empty CSVs.


In [7]:
mount_path = os.path.join(DATA_DIR, CAPSULE_MOUNT)

if os.path.isdir(mount_path):
    attached = set(os.listdir(mount_path))
    print(f'{len(attached)} entries in {CAPSULE_MOUNT}')

    # The mount may be keyed by asset name or by session id -- accept either.
    in_mount = (sessions.name.isin(attached)
                | sessions.session_id.isin(attached)
                | sessions.session_id.map(
                    lambda s: any(a.startswith(s) for a in attached)))
    print(f'{int(in_mount.sum())} of {len(sessions)} sessions present in the mount')

    if in_mount.any():
        sessions['in_capsule'] = in_mount
    else:
        print('WARNING: no session names matched the mount contents.')
        print('Sample mount entries:', sorted(attached)[:3])
        sessions['in_capsule'] = False
else:
    print(f'{mount_path} not attached to this capsule -- tables not filtered.')
    sessions['in_capsule'] = np.nan

sessions = sessions[sessions.in_capsule == True]

155 entries in Visual-Learning-SWDB
155 of 158 sessions present in the mount


In [8]:
# Drop test sessions with only 2 planes 

sessions_to_drop = ['multiplane-ophys_800792_2025-11-06_11-00-52_processed_2026-08-19_00-53-54', 'multiplane-ophys_800792_2025-11-07_11-04-07_processed_2026-08-19_00-53-56',
       'multiplane-ophys_800792_2025-11-11_09-26-07_processed_2026-08-19_00-53-56', 'multiplane-ophys_800792_2025-11-17_11-16-34_processed_2026-08-19_00-54-00',
       'multiplane-ophys_800995_2025-11-10_13-12-51_processed_2026-08-19_00-59-12', 'multiplane-ophys_800995_2025-11-12_12-30-02_processed_2026-08-19_00-59-22',
       'multiplane-ophys_800995_2025-11-13_10-16-12_processed_2026-08-19_01-00-17', 'multiplane-ophys_800995_2025-11-14_10-36-31_processed_2026-08-19_01-00-27']

print(len(sessions), 'sessions present in the mount')
sessions = sessions[sessions.name.isin(sessions_to_drop)==False]
print(len(sessions), 'after removing test sessions')

155 sessions present in the mount
147 after removing test sessions


### Sanity checks

docDB drops rows silently — it returns no error when an asset simply is not indexed.
Read these counts against what you expect from the processing batch; if a mouse is
short, re-run rather than assuming the data is missing.

In [9]:
print('sessions per mouse')
print(sessions.subject_id.value_counts().sort_index().to_string())

print('\nplanes per session')
print(sessions.n_planes.value_counts().sort_index().to_string())

print('\nsession types')
print(sessions.session_type.value_counts().to_string())

missing = set(CTL_MICE) - set(sessions.subject_id)
if missing:
    print(f'\nno sessions returned for: {sorted(missing)}')

sessions per mouse
subject_id
782149    24
788406    32
790322    24
800792    25
800995    22
804363    20

planes per session
n_planes
8    147

session types
session_type
TRAINING_1_gratings                      26
TRAINING_3_images_A_10uL_reward          19
STAGE_1                                  19
OPHYS_6_images_B                         15
OPHYS_1_images_A                         12
OPHYS_4_images_B                         12
TRAINING_2_gratings_flashed               9
TRAINING_4_images_A_training              7
TRAINING_0_gratings_autorewards_15min     7
TRAINING_5_images_A_epilogue              7
STAGE_0                                   7
TRAINING_5_images_A_handoff_ready         6
TRAINING_5_images_A_handoff_lapsed        1


### Write the session table

In [10]:
session_csv = f'{OUTPUT_DIR}/visual_learning_session_metadata.csv'
sessions.to_csv(session_csv, index=False)
print(f'{session_csv}  ({len(sessions)} rows, {sessions.shape[1]} columns)')

/data/metadata/visual_learning_session_metadata.csv  (147 rows, 24 columns)


---

## Z-drift QC (optional)

The session table
is already written above; this section only adds the per-plane z-drift table and fills
in the `planes_failing_zdrift` column, then rewrites the CSV.

QC lives on a different API version than the session metadata, so it uses its own
client. Two outputs:

- `visual_learning_zdrift_qc.csv` — one row per session x plane, with Pass/Fail
- `planes_failing_zdrift` back-filled onto the session table

Sessions whose processing generation predates the z-drift evaluation have no metric to
read; those stay `NA` rather than `0`, so a session with no QC is not mistaken for a
session that passed.

In [11]:
qc_client = MetadataDbClient(host=API_GATEWAY_HOST, 
                                    version="v1",
                                    database=DATABASE,
                                    collection=COLLECTION)

qc_aggregate = [
  {"$match": {"data_description.subject_id": {"$in": CTL_MICE},
              "name": {"$regex": "^multiplane-ophys_.*_processed_"},
              "quality_control.evaluations": {
                  "$elemMatch": {"name": {"$regex": "Z-drift"}}}}},
  {"$unwind": "$quality_control.evaluations"},
  {"$match": {"quality_control.evaluations.name": {"$regex": "Z-drift"}}},
  {"$unwind": "$quality_control.evaluations.metrics"},
  {"$project": {
      "name": 1,
      "subject_id": "$data_description.subject_id",
      "evaluation": "$quality_control.evaluations.name",
      "metric_name": "$quality_control.evaluations.metrics.name",
      "status_history": "$quality_control.evaluations.metrics.status_history",
  }},
]

qc = pd.DataFrame(qc_client.aggregate_docdb_records(pipeline=qc_aggregate))
print(f'{len(qc)} z-drift metric rows')

# status_history is append-only; the last entry is current.
qc['status'] = qc.status_history.map(
    lambda h: h[-1].get('status') if isinstance(h, list) and h else None)
qc['plane_name'] = qc.metric_name.str.extract(r'^([A-Za-z]+_\d+)')
qc['session_id'] = qc.name.str.extract(
    r'^(multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+)_processed_')
qc['processed_stamp'] = qc.name.str.extract(
    r'_processed_(\d{4}-\d{2}-\d{2}_[\d-]+)$')

qc = (qc.sort_values('processed_stamp')
        .drop_duplicates(['session_id', 'plane_name'], keep='last'))

qc = qc[['subject_id', 'session_id', 'plane_name', 'status',
         'evaluation', 'metric_name', 'processed_stamp', 'name']].sort_values(
    ['subject_id', 'session_id', 'plane_name']).reset_index(drop=True)

print(qc.status.value_counts().to_string())
print(f'{qc.session_id.nunique()} of {len(sessions)} sessions have z-drift QC')
qc.head()

2342 z-drift metric rows
status
Pass    828
Fail    116
124 of 147 sessions have z-drift QC


,subject_id,session_id,plane_name,status,evaluation,metric_name,processed_stamp,name
0,782149,multiplane-ophys_782149_2025-03-25_09-46-08,VISp_0,Pass,Z-drift Analysis,VISp_0 Z-drift Analysis - VISp_0,2026-08-19_00-32-51,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
1,782149,multiplane-ophys_782149_2025-03-25_09-46-08,VISp_1,Pass,Z-drift Analysis,VISp_1 Z-drift Analysis - VISp_1,2026-08-19_00-32-51,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
2,782149,multiplane-ophys_782149_2025-03-25_09-46-08,VISp_2,Pass,Z-drift Analysis,VISp_2 Z-drift Analysis - VISp_2,2026-08-19_00-32-51,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
3,782149,multiplane-ophys_782149_2025-03-25_09-46-08,VISp_3,Pass,Z-drift Analysis,VISp_3 Z-drift Analysis - VISp_3,2026-08-19_00-32-51,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
4,782149,multiplane-ophys_782149_2025-03-25_09-46-08,VISp_4,Pass,Z-drift Analysis,VISp_4 Z-drift Analysis - VISp_4,2026-08-19_00-32-51,multiplane-ophys_782149_2025-03-25_09-46-08_pr...


### Check the QC belongs to the same processing generation

The QC query keeps whichever generation carries a z-drift evaluation, which need not
be the generation in the session table. Confirm they agree before trusting the
back-filled column — a mismatch means a session's QC describes a different asset.

In [12]:
sessions['planes_failing_zdrift'] = pd.NA

stamps = sessions[['session_id', 'processed_stamp']].merge(
    qc.groupby('session_id').processed_stamp.first().rename('qc_stamp'),
    on='session_id', how='inner')

mismatched = stamps[stamps.processed_stamp != stamps.qc_stamp]
print(f'{len(stamps) - len(mismatched)} of {len(stamps)} match the session generation')
if len(mismatched):
    print(f'{len(mismatched)} sessions have QC from a different generation:')
    print(mismatched.to_string())

113 of 113 match the session generation


In [13]:
# Back-fill the session column: count of Fail planes, NA where QC is absent
fails = qc.status.eq('Fail').groupby(qc.session_id).sum()
have_qc = sessions.session_id.isin(qc.session_id)

sessions['planes_failing_zdrift'] = (
    sessions.session_id.map(fails).where(have_qc).astype('Int64'))

print(f'{int(have_qc.sum())} sessions with QC, {int((~have_qc).sum())} left NA')
print(sessions.planes_failing_zdrift.value_counts(dropna=False).sort_index().to_string())

113 sessions with QC, 34 left NA
planes_failing_zdrift
0       71
1       14
2       10
3        5
4        6
5        2
6        3
7        1
8        1
<NA>    34


### Write the QC table and the updated session table

In [14]:
qc_csv = f'{OUTPUT_DIR}/visual_learning_zdrift_qc.csv'
qc.to_csv(qc_csv, index=False)
print(f'{qc_csv}  ({len(qc)} rows, {qc.shape[1]} columns)')

# rewrite with planes_failing_zdrift filled in
sessions.to_csv(session_csv, index=False)
print(f'{session_csv}  (rewritten with planes_failing_zdrift)')

/data/metadata/visual_learning_zdrift_qc.csv  (944 rows, 8 columns)
/data/metadata/visual_learning_session_metadata.csv  (rewritten with planes_failing_zdrift)
